# Qualidade e cobertura dos dados (Gold)

Consolida, em uma tabela longa, as contagens e taxas que o TCC precisa declarar: perdas por filtro, cobertura de campos e validade dos segmentos (RN027). Apenas agregados; nenhum identificador.

* Destino: `cat_mobilidade_urbana_gold.db_frota.tb_qualidade_dados`

In [0]:
# No Databricks `spark` já existe; no VS Code o Databricks Connect cria uma sessão serverless.
try:
    spark  # noqa: F821
except NameError:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.serverless(True).getOrCreate()

from pyspark.sql import Window as W
from pyspark.sql import functions as F

# Todos os carimbos da fonte têm offset -03:00; datas e horas locais usam o fuso informado pelo proponente.
spark.conf.set("spark.sql.session.timeZone", "America/Sao_Paulo")

BRONZE = "cat_mobilidade_urbana_bronze.db_frota"
SILVER = "cat_mobilidade_urbana_silver.db_frota"
GOLD = "cat_mobilidade_urbana_gold.db_frota"


def segundos_entre(inicio, fim):
    """Diferença em segundos (com fração) entre dois timestamps."""
    return F.col(fim).cast("double") - F.col(inicio).cast("double")


def haversine_km(lat1, lon1, lat2, lon2):
    """Mesmo Haversine de src/tcc_frota/distance.py (R = 6371,0088 km).

    Linha reta entre dois pontos: estimativa geodésica, nunca rodagem observada (RN012).
    """
    la1, lo1, la2, lo2 = (F.radians(F.col(c)) for c in (lat1, lon1, lat2, lon2))
    h = F.pow(F.sin((la2 - la1) / 2), 2) + F.cos(la1) * F.cos(la2) * F.pow(F.sin((lo2 - lo1) / 2), 2)
    return F.lit(6371.0088) * 2 * F.asin(F.least(F.lit(1.0), F.sqrt(h)))


def pseudonimo(coluna):
    """Pseudônimo determinístico (SHA-256 truncado) para tabelas Gold; não é anonimização."""
    return F.substring(F.sha2(F.concat(F.lit("tcc-frota|"), F.col(coluna).cast("string")), 256), 1, 16)


def gravar(df, tabela, comentario):
    """Sobrescreve a tabela Delta de forma idempotente e registra o comentário."""
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(tabela)
    spark.sql(f"COMMENT ON TABLE {tabela} IS '{comentario}'")
    print(f"{tabela}: {spark.table(tabela).count():,} linhas")

In [0]:
metricas = []


def registrar(camada, tabela, metrica, valor, descricao):
    metricas.append((camada, tabela, metrica, float(valor) if valor is not None else None, descricao))


def taxa(df, condicao):
    # Condição nula conta como falsa, para a taxa ser sobre todas as linhas.
    linha = df.agg(F.avg(F.coalesce(F.expr(condicao), F.lit(False)).cast("double")).alias("v")).first()
    return linha["v"]


for tabela in ("tb_solicitacoes", "tb_ofertas", "tb_sumarios"):
    registrar("bronze", tabela, "linhas", spark.table(f"{BRONZE}.{tabela}").count(), "Documentos ingeridos")

sol = spark.table(f"{SILVER}.tb_solicitacoes")
registrar("silver", "tb_solicitacoes", "linhas", sol.count(), "Solicitacoes apos dedup por codigo")
registrar("silver", "tb_solicitacoes", "taxa_atendida", taxa(sol, "fl_atendida"), "CorridaEncerrada / solicitacoes")
registrar("silver", "tb_solicitacoes", "taxa_sem_log_status", taxa(sol, "fl_sem_log_status"), "Sem historico logStatus")
registrar("silver", "tb_solicitacoes", "taxa_tempos_ordenados_atendidas",
          taxa(sol.where("fl_atendida"), "fl_tempos_ordenados"), "Atendidas com registro<=aceite<=chegada<=inicio<fim")
registrar("silver", "tb_solicitacoes", "taxa_agendamento", taxa(sol, "tipo_solicitacao = 'Agendamento'"), "Solicitacoes agendadas")

pos = spark.table(f"{SILVER}.tb_posicoes_motoristas")
registrar("silver", "tb_posicoes_motoristas", "linhas", pos.count(), "Motorista x snapshot")
registrar("silver", "tb_posicoes_motoristas", "taxa_coordenada_valida", taxa(pos, "fl_coordenada_valida"), "Coordenada no Brasil e diferente de 0,0")
registrar("silver", "tb_posicoes_motoristas", "taxa_posicao_atual", taxa(pos, "fl_posicao_atual"), "Idade da posicao entre -60 e 300 s")
registrar("silver", "tb_posicoes_motoristas", "taxa_estado_nao_mapeado", taxa(pos, "estado = 'NAO_MAPEADO'"), "Codigo de status fora do mapa empirico")
registrar("silver", "tb_posicoes_motoristas", "taxa_bbox_grande_vitoria", taxa(pos, "fl_bbox_grande_vitoria"), "Dentro do retangulo aproximado")
registrar("silver", "tb_posicoes_motoristas", "motoristas_distintos", pos.select("id_motorista").distinct().count(), "Ofertantes distintos")

des = spark.table(f"{SILVER}.tb_deslocamentos_motoristas")
registrar("silver", "tb_deslocamentos_motoristas", "linhas", des.count(), "Trechos entre posicoes consecutivas")
registrar("silver", "tb_deslocamentos_motoristas", "taxa_trecho_valido", taxa(des, "fl_passo_valido"), "Intervalo <= 300 s e velocidade <= 150 km/h")
registrar("silver", "tb_deslocamentos_motoristas", "taxa_lacuna_temporal", taxa(des, "motivo_invalido = 'LACUNA_TEMPORAL'"), "Intervalo > 300 s")
registrar("silver", "tb_deslocamentos_motoristas", "taxa_velocidade_implausivel", taxa(des, "motivo_invalido = 'VELOCIDADE_IMPLAUSIVEL'"), "Velocidade > 150 km/h")

sum_ = spark.table(f"{SILVER}.tb_sumarios")
registrar("silver", "tb_sumarios", "taxa_consistente_com_posicoes", taxa(sum_, "fl_consistente_com_posicoes"), "Contadores iguais as posicoes do snapshot")

cor = spark.table(f"{SILVER}.tb_corridas")
registrar("silver", "tb_corridas", "linhas", cor.count(), "Corridas encerradas")
for flag in ("fl_aceite_confiavel", "fl_origem_confiavel", "fl_destino_confiavel", "fl_corrida_espacial_valida"):
    registrar("silver", "tb_corridas", f"taxa_{flag[3:]}", taxa(cor, flag), "Marco com GPS do motorista a ate 120 s")

seg = spark.table(f"{GOLD}.tb_km_vazio_segmentos")
registrar("gold", "tb_km_vazio_segmentos", "linhas", seg.count(), "Pares de corridas consecutivas do mesmo motorista")
registrar("gold", "tb_km_vazio_segmentos", "taxa_segmento_valido", taxa(seg, "fl_segmento_valido"), "Mesma sessao e posicoes confiaveis")
registrar("gold", "tb_km_vazio_segmentos", "taxa_mesma_sessao", taxa(seg, "fl_mesma_sessao"), "Sem lacuna de GPS nem INDISPONIVEL")
registrar("gold", "tb_km_vazio_segmentos", "taxa_aceite_antes_fim_anterior", taxa(seg, "fl_aceite_antes_fim_anterior"), "Aceite durante a corrida anterior")
registrar("gold", "tb_corridas_replay", "linhas", spark.table(f"{GOLD}.tb_corridas_replay").count(), "Corridas para replay")

qualidade = spark.createDataFrame(
    metricas, "camada string, tabela string, metrica string, valor double, descricao string"
).withColumn("_ts_processamento", F.current_timestamp())
gravar(qualidade, f"{GOLD}.tb_qualidade_dados", "Metricas agregadas de cobertura, perdas por filtro e validade por tabela.")

for m in metricas:
    print(m[:4])